# ANPR: train YOLOv8 on the leak-free split v2 (Kaggle GPU)

Part of `number-plate-vehicle-detection`, branch `upgrade-v2`. See `docs/audit.md` in the repo.

**Why this notebook exists:** the original model (98.5 % val mAP) was evaluated on a split where 82.6 % of val and
86.0 % of test images had a near-duplicate in train. 879 of 1,782 merged images were duplicate copies. Split v2
(`tools/make_group_split.py`) removed the copies and keeps each scene in one split: train 629 / val 151 / test 123,
0 cross-split near-duplicates. This notebook retrains on v2 and reports **held-out test** metrics.

**Setup (once):**
1. Settings → Accelerator: **GPU T4 x2** or **GPU P100**. Internet: **On** (needs a phone-verified Kaggle account).
2. Add data → your private dataset made from `final_v2.zip`.
3. Run all cells to check it works, then **Save Version → Save & Run All (Commit)** so it keeps running with the browser closed.
4. Download `/kaggle/working/anpr_v2_results.zip` from the version's Output tab.

Hyper-parameters match `scripts/train.py` (and the original baseline), except model size, batch, and `amp=True` (T4/P100 have fast FP16).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q ultralytics==8.4.48

In [ ]:
import json
import shutil
import glob
import time
from pathlib import Path
import yaml

# Find the uploaded dataset (folder that holds data.yaml + images/train), wherever Kaggle unpacked it
cands = [Path(p).parent for p in glob.glob('/kaggle/input/**/data.yaml', recursive=True)
         if (Path(p).parent / 'images' / 'train').is_dir()]
assert cands, 'data.yaml with images/train not found under /kaggle/input - check the dataset upload'
SRC = cands[0]
DATA = Path('/kaggle/working/final_v2')          # writable copy (Ultralytics writes label caches)
if not DATA.exists():
    shutil.copytree(SRC, DATA)
cfg = yaml.safe_load(open(DATA / 'data.yaml'))
cfg['path'] = str(DATA)
(DATA / 'data_kaggle.yaml').write_text(yaml.safe_dump(cfg, sort_keys=False))
for s in ['train', 'val', 'test']:
    n_img = len(list((DATA / 'images' / s).iterdir()))
    n_lbl = len(list((DATA / 'labels' / s).glob('*.txt')))
    print(f'{s:5s} images {n_img:4d}  labels {n_lbl:4d}')
print('classes:', cfg['names'])
# Expected: train 629 / val 151 / test 123

In [ ]:
from ultralytics import YOLO

HYP = dict(  # same as scripts/train.py / original baseline (results/baseline_yolov8l/args.yaml)
    optimizer='AdamW', lr0=0.0005, lrf=0.01, weight_decay=0.0005, momentum=0.937, cos_lr=True,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, degrees=10.0, translate=0.1, scale=0.5, shear=2.0,
    perspective=0.0005, flipud=0.0, fliplr=0.5, mosaic=1.0, mixup=0.15, copy_paste=0.1, close_mosaic=10,
    seed=0, deterministic=True, pretrained=True, patience=30, plots=True, val=True, save_period=-1,
)
RUNS = Path('/kaggle/working/runs')

def train_and_test(model_name, imgsz, batch, epochs=100):
    name = f'v2_{Path(model_name).stem}_{imgsz}'
    t0 = time.time()
    model = YOLO(model_name)
    model.train(data=str(DATA / 'data_kaggle.yaml'), imgsz=imgsz, batch=batch, epochs=epochs,
                device=0, workers=4, amp=True, cache='ram', project=str(RUNS), name=name, exist_ok=True, **HYP)
    minutes = (time.time() - t0) / 60
    best = YOLO(RUNS / name / 'weights' / 'best.pt')
    out = {'model': model_name, 'imgsz': imgsz, 'batch': batch, 'train_minutes': round(minutes, 1)}
    for split in ['val', 'test']:     # val chose best.pt; TEST is the held-out number to report
        m = best.val(data=str(DATA / 'data_kaggle.yaml'), split=split, imgsz=imgsz, batch=batch, device=0,
                     plots=True, project=str(RUNS / name), name=f'eval_{split}', exist_ok=True)
        out[split] = {'precision': float(m.box.mp), 'recall': float(m.box.mr),
                      'mAP50': float(m.box.map50), 'mAP50_95': float(m.box.map),
                      'per_class': {best.names[int(c)]: {'mAP50': float(m.box.ap50[i]), 'mAP50_95': float(m.box.ap[i])}
                                    for i, c in enumerate(m.box.ap_class_index)}}
    (RUNS / name / 'metrics_v2.json').write_text(json.dumps(out, indent=2))
    print(json.dumps({k: v for k, v in out.items() if k != 'val'}, indent=2))
    return out

results = {}

## Run 1: YOLOv8s @ 960 (small; the one you'd deploy on the laptop). About 30–60 min on a T4.

In [ ]:
results['yolov8s_960'] = train_and_test('yolov8s.pt', imgsz=960, batch=16)

## Run 2: YOLOv8l @ 960 (same size as the original baseline → fair before/after). About 1.5–3 h on a T4.
If you hit CUDA out-of-memory, change `batch=8` to `batch=4`.

In [ ]:
results['yolov8l_960'] = train_and_test('yolov8l.pt', imgsz=960, batch=8)

In [ ]:
# Summary table + package everything small enough to download (weights, curves, csv, metrics)
rows = []
for k, r in results.items():
    t = r['test']
    rows.append(f"| {k} | {r['train_minutes']} | {r['val']['mAP50']:.3f} | {t['precision']:.3f} | {t['recall']:.3f} | {t['mAP50']:.3f} | {t['mAP50_95']:.3f} |")
summary = '\n'.join(['| run | train min | val mAP50 | test P | test R | test mAP50 | test mAP50-95 |',
                      '|---|---|---|---|---|---|---|'] + rows)
print(summary)
(RUNS / 'summary_v2.md').write_text(summary + '\n')

pack = Path('/kaggle/working/pack')
if pack.exists():
    shutil.rmtree(pack)
for run in RUNS.iterdir():
    if not run.is_dir():
        continue
    dst = pack / run.name
    for f in run.rglob('*'):
        if f.is_file() and (f.suffix in {'.csv', '.yaml', '.json', '.png', '.md'} or f.name == 'best.pt') \
                and not f.name.startswith('train_batch'):
            (dst / f.relative_to(run)).parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dst / f.relative_to(run))
shutil.copy2(RUNS / 'summary_v2.md', pack / 'summary_v2.md')
shutil.make_archive('/kaggle/working/anpr_v2_results', 'zip', pack)
print('Download /kaggle/working/anpr_v2_results.zip from the Output tab')